# Auditoría de sesgo demográfico en detección de deepfakes (Colab)

**Proyecto Ley Olimpia · Global South AI Safety Hackathon — LatAm**

Pregunta única: *¿un detector público de deepfakes comete más errores con rostros de
ciertos grupos, dejando esos deepfakes sin detectar?* Métrica: **tasa de falsos
negativos (FNR) por grupo**. Si a un grupo se le escapan más, ese grupo queda menos
protegido — ahí conecta con la Ley Olimpia.

Este notebook corre el flujo completo en Colab con GPU gratis:

1. Setup (clonar repo + instalar dependencias)
2. Elegir y preparar el dataset de caras real/fake
3. Correr el detector → `data/preds.csv`
4. Etiquetar demografía con FairFace → `data/labels.csv`
5. Analizar el sesgo (FNR, AUROC, chi², brecha con IC95%) + figuras

**Con la extensión Colab en Cursor/VS Code:** abre este notebook → **Select Kernel →
Colab → Auto Connect** (o un servidor con GPU). Si ya te autenticaste, deberías ver
"Colab" en la lista de kernels. El código corre en los servidores de Google, no en tu Mac.

**Ética (innegociable):** solo rostros benignos de datasets públicos, solo adultos
(se filtra edad < 20), nada de contenido íntimo ni de generar deepfakes. La demografía
la *percibe* un modelo (proxy), no la declara la persona: es una limitación que va en la
sección de doble uso del reporte.

## 0. Verificación rápida (ejecuta primero)

Confirma que el kernel de Colab está conectado y que hay GPU disponible.
Si esta celda falla o `CUDA disponible: False`, vuelve a **Select Kernel → Colab → Auto Connect**
y elige un runtime con GPU.

In [ ]:
import os, sys, platform

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Directorio actual:", os.getcwd())

try:
    import torch
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("⚠️ Sin GPU: reconecta el kernel Colab con runtime GPU.")
except ImportError:
    print("torch aún no instalado (normal antes del paso de dependencias)")

print("\n✓ Kernel Colab respondiendo. Sigue con la celda de setup.")

## 1. Setup: clonar el repo e instalar dependencias

Clona el repo público (ya incluye `src/` y `requirements.txt`). Si prefieres no usar git,
puedes subir la carpeta `src/` manualmente y saltarte la celda de clonado.

In [ ]:
import os

# === EDITA ESTO ===
REPO_URL = "https://github.com/angelpineda-clients/ley-olimpia.git"
PROJECT_DIR = "/content/ley-olimpia"
# ==================

if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
else:
    print("El repo ya está clonado; haciendo pull...")
    !cd $PROJECT_DIR && git pull --ff-only

os.chdir(PROJECT_DIR)
print("Directorio de trabajo:", os.getcwd())
!ls -la

In [ ]:
# Dependencias. En Colab, torch ya viene instalado con CUDA; instalamos el resto.
!pip install -q transformers deepface pandas numpy scipy scikit-learn matplotlib pillow kagglehub

# Verificación rápida de GPU (opcional pero recomendado).
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Elegir y preparar el dataset

El objetivo es dejar imágenes en `data/images/real/` y `data/images/fake/`. Tienes
tres caminos; **ejecuta solo el que elijas**:

- **Opción A — Kaggle "140k Real and Fake Faces" (recomendado):** rápido, sin EULA.
  Reales = fotos Flickr/FFHQ, falsas = caras StyleGAN (contenido benigno, adultos).
  Estructura: `real_vs_fake/real-vs-fake/{train,valid,test}/{real,fake}`.
- **Opción B — Hugging Face:** cualquier dataset de "real vs fake faces". Revisa en la
  ficha los nombres de columnas/splits y la licencia antes de usar.
- **Opción C — Subir tus propias imágenes:** si ya las tienes organizadas.

Ajusta `SAMPLE_PER_CLASS` para el MVP (unos cientos a 1-2 mil por clase; submuestrea si
va lento). **Revisa la ficha del dataset y quédate solo con adultos.**

In [ ]:
# Helper común: copia un submuestreo de imágenes hacia data/images/{real,fake}.
import random, shutil
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SAMPLE_PER_CLASS = 400  # imágenes por clase para el MVP (ajustable)

DEST = Path("data/images")
(DEST / "real").mkdir(parents=True, exist_ok=True)
(DEST / "fake").mkdir(parents=True, exist_ok=True)


def list_images(folder):
    folder = Path(folder)
    return [p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS]


def populate_class(src_folder, klass, n=SAMPLE_PER_CLASS, seed=0):
    """Copia hasta n imágenes de src_folder a data/images/<klass>/."""
    imgs = list_images(src_folder)
    if not imgs:
        raise SystemExit(f"No se hallaron imágenes en {src_folder}")
    random.Random(seed).shuffle(imgs)
    imgs = imgs[:n]
    dest = DEST / klass
    for i, p in enumerate(imgs):
        shutil.copy(p, dest / f"{klass}_{i:05d}{p.suffix.lower()}")
    print(f"[ok] {len(imgs)} imágenes -> {dest}")


def summary():
    nr = len(list(Path("data/images/real").glob("*")))
    nf = len(list(Path("data/images/fake").glob("*")))
    print(f"Total preparado -> real: {nr} | fake: {nf}")

print("Helpers listos. Ejecuta UNA de las opciones A/B/C abajo.")

In [ ]:
# === OPCIÓN A: Kaggle "140k Real and Fake Faces" (recomendada) ===
# kagglehub descarga el dataset (puede pedir credenciales de Kaggle la 1a vez:
# crea un token en kaggle.com -> Settings -> API, y súbelo, o usa kagglehub.login()).
import kagglehub
from pathlib import Path

ds_path = Path(kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces"))
print("Descargado en:", ds_path)

# La estructura típica es real-vs-fake/{train,valid,test}/{real,fake}.
# Usamos el split 'valid' para ir rápido; cambia a 'test' o 'train' si quieres más.
base = next(ds_path.rglob("valid"), None) or ds_path
real_src = base / "real"
fake_src = base / "fake"
print("real_src:", real_src, "| existe:", real_src.exists())
print("fake_src:", fake_src, "| existe:", fake_src.exists())

populate_class(real_src, "real")
populate_class(fake_src, "fake")
summary()

In [ ]:
# === OPCIÓN B: Hugging Face (alternativa) ===
# Ejemplo genérico: ajusta DATASET_ID, el split y el nombre de la columna de etiqueta.
# Revisa la ficha del dataset para saber cómo distingue real vs fake.
#
# from datasets import load_dataset
# from pathlib import Path
#
# DATASET_ID = "AJUSTA/real-vs-fake-faces"   # <-- pon el id real
# SPLIT = "train"
# LABEL_COL = "label"        # columna con la clase
# IMAGE_COL = "image"        # columna con la imagen (PIL)
# FAKE_VALUES = {1, "fake", "ai"}  # qué valores cuentan como 'fake'
#
# ds = load_dataset(DATASET_ID, split=SPLIT)
# n_real = n_fake = 0
# for ex in ds:
#     is_fake = ex[LABEL_COL] in FAKE_VALUES
#     if is_fake and n_fake >= SAMPLE_PER_CLASS: continue
#     if not is_fake and n_real >= SAMPLE_PER_CLASS: continue
#     klass = "fake" if is_fake else "real"
#     idx = n_fake if is_fake else n_real
#     ex[IMAGE_COL].convert("RGB").save(Path("data/images")/klass/f"{klass}_{idx:05d}.jpg")
#     if is_fake: n_fake += 1
#     else: n_real += 1
#     if n_real >= SAMPLE_PER_CLASS and n_fake >= SAMPLE_PER_CLASS: break
# summary()
print("Opción B comentada: descoméntala y ajusta el DATASET_ID si usas Hugging Face.")

In [ ]:
# === OPCIÓN C: subir tus propias imágenes ===
# Sube dos zips (real.zip y fake.zip) o usa files.upload(); luego acomódalas:
#
# from google.colab import files
# up = files.upload()   # selecciona real.zip y fake.zip
# !unzip -oq real.zip -d /content/raw_real
# !unzip -oq fake.zip -d /content/raw_fake
# populate_class("/content/raw_real", "real")
# populate_class("/content/raw_fake", "fake")
# summary()
print("Opción C comentada: úsala si ya tienes tus imágenes organizadas.")

## 3. Correr el detector de deepfakes → `data/preds.csv`

Usa el modelo por defecto (`prithivMLmods/Deep-Fake-Detector-Model`). Para robustez,
puedes repetir con `--model prithivMLmods/Deepfake-Detect-Siglip2`.

In [ ]:
!python src/run_detector.py --model prithivMLmods/Deep-Fake-Detector-Model

import pandas as pd
preds = pd.read_csv("data/preds.csv")
print(preds.shape)
preds.head()

## 4. Etiquetar demografía (FairFace vía deepface) → `data/labels.csv`

Estima edad, género y raza por rostro y **filtra menores** (`--min_age 20`). La raza
incluye la categoría `latino hispanic`, que da el ángulo LatAm.

In [ ]:
!python src/label_demographics.py --images data/images --min_age 20

import pandas as pd
labels = pd.read_csv("data/labels.csv")
print(labels.shape)
print("\nDistribución por raza:\n", labels["race"].value_counts())
print("\nDistribución por género:\n", labels["gender"].value_counts())
labels.head()

## 5. Analizar el sesgo: FNR, AUROC, chi², brecha con IC95% + figuras

Eje principal **raza** (lente LatAm) y eje secundario **género** (chequeo gratis con el
mismo pipeline).

In [ ]:
# Eje principal: raza/etnia (lente LatAm)
!python src/analyze_bias.py --group race

# Eje secundario: género
!python src/analyze_bias.py --group gender

In [ ]:
# Mostrar tablas y figuras inline
import pandas as pd
from IPython.display import Image, display

for group in ["race", "gender"]:
    print(f"\n===== Sesgo por {group} =====")
    display(pd.read_csv(f"data/outputs/bias_{group}.csv"))
    display(Image(filename=f"data/outputs/bias_{group}.png"))

## Cómo leer los resultados

- **FNR alto en un grupo** = a ese grupo se le escapan más deepfakes → menos protección.
- **Brecha de FNR (peor − mejor) con IC95%:** si el intervalo **no cruza 0**, hay
  evidencia de diferencia entre grupos. Si lo cruza, no se puede afirmar sesgo con esta
  muestra (recuerda: la literatura está dividida; ambos resultados son publicables).
- **chi²:** `p < 0.05` sugiere que acertar/fallar en los fakes depende del grupo.
- Cuida los tamaños de muestra por grupo (`n_fake`); el análisis ignora grupos con < 5
  fakes para no reportar tasas inestables. Si `latino hispanic` queda con `n` chico,
  súbelo aumentando `SAMPLE_PER_CLASS` o usando otro split del dataset.

**Para el reporte:** guarda `data/outputs/bias_race.csv` y `bias_race.png` (y los de
género). Documenta en la sección de doble uso que la demografía es percibida por un
modelo (proxy) y valida a mano ~40 etiquetas para reportar el acuerdo.